# 04 — Train the residual dilated TCN experiment

**Viraat Chaudhary · DATA 266 Lab 1 · Team 15 · Yelp Polarity**

Use **Colab A100**. Notebook 01 has already completed; do not rerun it. This notebook restores the approved processed ZIP from your existing Drive backup. No dataset download or vocabulary fitting is performed. All weights start randomly; checkpoint selection and early stopping use validation only.

Complete notebook 03 first. The default TCN uses six residual blocks, two causal convolutions per block, kernel 3 and dilations 1–32. Its receptive field is 253 tokens; global masked pooling aggregates positions across the capped review. This is not a guarantee of better metrics.

Run top-to-bottom. Save the executed notebook using **File → Download → Download .ipynb**. Use the exact filename provided. Outputs persist in your own Drive run folder. Source code is assistant-assisted; student explanations and analysis must be your own.


## Restore the package and packages

In [1]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
from google.colab import drive, files

# Set these once; keep the same RUN_NAME for notebooks 02–07 and resumed sessions.
DRIVE_BASE_RELATIVE = 'DATA266_Lab1_Task2/Viraat_Chaudhary'
RUN_NAME = 'task2_models_rnn_lstm_tcn_v1'
PACKAGE_ZIP_NAME = 'Viraat_Task2_Run_Package.zip'
PACKAGE_ID = 'viraat_yelp_rnn_lstm_tcn_v1'

drive.mount('/content/drive', force_remount=False)
DRIVE_BASE = Path('/content/drive/MyDrive') / DRIVE_BASE_RELATIVE
ROOT = DRIVE_BASE / RUN_NAME / 'task2_sentiment/Viraat_Chaudhary'
CACHE_ROOT = Path('/content/viraat_task2_data')
if not ROOT.exists():
    archive = DRIVE_BASE / PACKAGE_ZIP_NAME
    if not archive.exists():
        print('Upload Viraat_Task2_Run_Package.zip (the code package, not a dataset ZIP).')
        uploaded = files.upload()
        if PACKAGE_ZIP_NAME not in uploaded:
            raise FileNotFoundError('Select the exact package ZIP filename.')
        archive = Path('/content') / PACKAGE_ZIP_NAME
        archive.write_bytes(uploaded[PACKAGE_ZIP_NAME])
    extraction = Path('/content/viraat_task2_package_extract')
    extraction.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        for info in bundle.infolist():
            if not (extraction / info.filename).resolve().is_relative_to(extraction.resolve()):
                raise ValueError('Invalid package ZIP path.')
        if bundle.testzip() is not None:
            raise ValueError('Package ZIP CRC failed.')
        bundle.extractall(extraction)
    manifest = json.loads((extraction / 'PACKAGE_CONTENT_MANIFEST.json').read_text())
    if manifest['package_id'] != PACKAGE_ID:
        raise ValueError('Incorrect package identity.')
    for item in manifest['files']:
        path = extraction / item['path']
        if hashlib.sha256(path.read_bytes()).hexdigest() != item['sha256']:
            raise ValueError('Package integrity failed: ' + item['path'])
    source = extraction / 'task2_sentiment/Viraat_Chaudhary'
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, ROOT)
    (ROOT / 'package_installation.json').write_text(json.dumps({'package_id': PACKAGE_ID,
                                                              'package_manifest_sha256': hashlib.sha256(
        (extraction / 'PACKAGE_CONTENT_MANIFEST.json').read_bytes()).hexdigest()}, indent=2) + '\n')
else:
    installed = json.loads((ROOT / 'package_installation.json').read_text())
    if installed['package_id'] != PACKAGE_ID:
        raise ValueError('This RUN_NAME belongs to a different package.')

required = {'numpy': 'numpy==2.1.3', 'scipy': 'scipy>=1.14,<2',
            'sklearn': 'scikit-learn==1.6.1', 'nltk': 'nltk==3.9.1',
            'datasets': 'datasets==4.8.5', 'matplotlib': 'matplotlib==3.10.0',
            'pandas': 'pandas>=2.2,<4', 'tqdm': 'tqdm==4.67.3'}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Use a Colab runtime with PyTorch installed.')
# Keep the tokenizer version consistent with the approved snapshot.
import importlib.metadata
if importlib.metadata.version('nltk') != '3.9.1':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nltk==3.9.1'])

source_dir = str(ROOT / 'src')
sys.path.insert(0, source_dir)
for name in ('runtime_utils', 'yelp_data', 'data_runtime', 'models', 'metrics',
             'training', 'evaluation', 'review', 'reporting', 'evaluator'):
    module = sys.modules.get(name)
    if module is not None and Path(getattr(module, '__file__', '')).parent != ROOT / 'src':
        del sys.modules[name]
from data_runtime import verify_preprocessing
verify_preprocessing(ROOT)
import torch
import pandas as pd
print('Member folder:', ROOT)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Approved preprocessing restored; original notebook 01 is preserved.')


Mounted at /content/drive
Member folder: /content/drive/MyDrive/DATA266_Lab1_Task2/Viraat_Chaudhary/task2_models_rnn_lstm_tcn_v1/task2_sentiment/Viraat_Chaudhary
GPU: NVIDIA A100-SXM4-40GB
Approved preprocessing restored; original notebook 01 is preserved.


In [2]:
from runtime_utils import read_json
for prior in ['baseline', 'experiment_1']:
    manifest = read_json(ROOT / f'outputs/metrics/{prior}_training_manifest.json')
    if manifest['status'] != 'completed':
        raise RuntimeError('Complete the previous training notebook first.')


## Restore and verify the approved development arrays

In [3]:
from data_runtime import restore_development
DEV, PROCESSED_DIR = restore_development(ROOT, DRIVE_BASE, CACHE_ROOT)
print('Train:', len(DEV['train']['labels']), '| validation:', len(DEV['validation']['labels']))


Train: 504000 | validation: 56000


## Train or resume

An epoch-boundary checkpoint supports recovery after a disconnection. Completed runs are reused. Interrupted partial epochs are restarted; their raw records are retained and the timing scope is documented.

In [4]:
from training import train_model
TRAINING_RESULT = train_model(ROOT, 'experiment_2', DEV)
pd.DataFrame([TRAINING_RESULT]).T


experiment_2 epoch 1:   0%|          | 0/985 [00:00<?, ?it/s]

Epoch 1: validation macro-F1=0.92856; training phase=55.6s; best epoch=1.


experiment_2 epoch 2:   0%|          | 0/985 [00:00<?, ?it/s]

Epoch 2: validation macro-F1=0.93762; training phase=53.1s; best epoch=2.


experiment_2 epoch 3:   0%|          | 0/985 [00:00<?, ?it/s]

...

experiment_2 epoch 4:   0%|          | 0/985 [00:00<?, ?it/s]

Epoch 4: validation macro-F1=0.94336; training phase=53.0s; best epoch=4.


experiment_2 epoch 5:   0%|          | 0/985 [00:00<?, ?it/s]

Epoch 5: validation macro-F1=0.94367; training phase=53.0s; best epoch=5.


,0
status,completed
model,experiment_2
identity,d19da4ef0b3e1a59e01a8a6d144660f25397d3816971d1...
config,configs/experiment_2.json
config_sha256,30507082a8277fd5648ab005fcfdd9e0823d3a09d48200...
data_contract,e3010a4d701170a1bb7e0378fd1049ac391c5031b40452...
implementation_fingerprint,6c9bf154d4817e56ab49065c1fe1a3ab0fa4146f451fe4...
checkpoint,checkpoints/experiment_2_best.pt
checkpoint_sha256,4bdac5813be95e42904cf2f733e2c0675528dd7ef9ef96...
selected_epoch,5


## Save stage evidence

Files are saved in Drive. The small ZIP excludes model weights; the final notebook exports all selected weights. Download this executed notebook before opening the next one.

In [5]:
from reporting import refresh_reports, export_artifacts
refresh_reports(ROOT)
STAGE_ZIP = export_artifacts(ROOT, '04_Training_Evidence')
print('Stage evidence:', STAGE_ZIP)
print('Download the executed notebook with File → Download → Download .ipynb.')


Stage evidence: /content/drive/MyDrive/DATA266_Lab1_Task2/Viraat_Chaudhary/task2_models_rnn_lstm_tcn_v1/task2_sentiment/Viraat_Chaudhary/exports/Viraat_Task2_04_Training_Evidence.zip
Download the executed notebook with File → Download → Download .ipynb.


## Freeze all three checkpoints before official-test access

In [6]:
from training import freeze_models
FROZEN = freeze_models(ROOT)
print('Frozen models:', list(FROZEN['models']))
print('Manual-review model selected using validation:', FROZEN['manual_review_model'])


Frozen models: ['baseline', 'experiment_1', 'experiment_2']
Manual-review model selected using validation: experiment_1
